# 답변 프롬프트 비교: 규정 없음 거절 (예전 vs 새) · Colab GPU

**보려는 것**: 답변 프롬프트의 규칙 2·3번을 고친 것(backend 브랜치 `fix/refusal-prompt`, 합쳐진 뒤에는 main)이 '규정 없음' 질문을 제대로 거절하게 하는지, 그러면서 **답할 수 있는 질문을 괜히 거절하지 않는지**.

| | 규칙 2 | 규칙 3 |
| --- | --- | --- |
| 예전 (backend `6d9edba`, 11/18 평가 때) | 근거에서 답을 찾을 수 없으면 "규정에서 확인되지 않아요. 인사팀에 확인해 주세요." | 질문의 전제가 근거와 다르면 먼저 "아니요"라고 바로잡기 |
| 새 | 질문이 묻는 **바로 그 내용**이 근거에 **직접** 있을 때만 답하고, 비슷한 조항으로 추론해 결론 내지 않기. 일부만 있으면 있는 부분만 | 근거가 전제와 **반대되는 내용을 직접** 말할 때만 "아니요" |

방식 (11/18 Oracle 평가와 같음)
- 문항: 골든셋 dev + test 중 **본인 데이터·계산이 필요 없는 규정 질문 189개** (일반 171 · 규정 없음 10 · 틀린 전제 4 · 포괄 4) + **퇴직연금 골든셋 9개**(A 규정 5 · D 거절·틀린 전제 4, `eval/golden/pension`). 퇴직연금 B·C는 본인 퇴직연금 데이터가 DB에 아직 없어서 빼요. 퇴직연금 문항 중 본인 정보가 필요한 일부(예: "현재 DB 가입자라")는 조회 없이 답해서 두 프롬프트 모두 손해를 봐요
- 근거: `eval_golden.oracle_context` = 정답 조문 + 검색 상위 비슷한 오답 조문으로 5개 (규정 없음 문항은 검색 상위 5개). 검색은 11/18과 같은 하이브리드 + 리랭커(가점 0.2)
- 답변: backend `nodes._llm_answer` 그대로, `SYSTEM_PROMPT`만 예전/새로 바꿔 끼움 → `normalize_no_evidence`
- 채점: 같은 LLM으로 `eval_golden` 판정(JUDGE_SYSTEM) → `eval_golden.cmd_score` 그대로 (정확도, 규정없음 거절률, 잘못된 거절률 …)
- 거절 판정이 두 가지예요: `eval_golden`의 **넓은 기준**("인사팀 확인" 같은 말만 들어가도 거절로 셈)과 **좁은 기준**(답이 "규정에서 확인되지 않아요"로 시작해 거절로 처리된 경우만)

런타임 → 런타임 유형 변경 → **A100 이상 권장**(Qwen3.5-27B-FP8). GPU가 39GB보다 작으면(L4 등) 27B가 안 들어가서 **Qwen3.5-9B로 돌리고 결과에 크게 표시**해요 — 예전/새 비교용 근사치예요.

In [ ]:
!pip install -q -U "sentence-transformers>=3.4" rank_bm25 kiwipiepy openai python-dotenv httpx langchain-openai langchain-core

In [ ]:
import os, sys, json, re, time, math, subprocess, hashlib, importlib.util, shutil
from pathlib import Path
from collections import Counter, defaultdict
from types import SimpleNamespace
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch

ROOT = Path("/content/hr_chatbot")
ROOT.mkdir(parents=True, exist_ok=True)
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if HAS_GPU else 0
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "없음", f"{GPU_GB:.0f}GB")

CFG = {
    # 검색: 11/18 임직원 평가 summary.json 의 cfg 그대로
    "embed_model": "BAAI/bge-m3", "rerank_model": "BAAI/bge-reranker-v2-m3",
    "k_dense": 30, "k_bm25": 30, "k_candidates": 40, "rrf_c": 60,
    "bm25_weight": 1.0, "dense_weight": 1.0, "law_weight": 0.6, "rewrite_bm25": True,
    "internal_bonus": 0.2, "attach_appendix": True, "max_chars_per_article": 1200,
    # LLM
    "llm_model": "Qwen/Qwen3.5-27B-FP8" if GPU_GB >= 39 else "Qwen/Qwen3.5-9B",
    "llm_port": 8000, "thinking": False, "llm_max_tokens": 1024, "llm_concurrency": 16,
}
CFG["served_name"] = "qwen3.5-27b" if "27B" in CFG["llm_model"] else "qwen3.5-9b"
if "9B" in CFG["llm_model"]:
    print("⚠️⚠️ GPU가 작아 Qwen3.5-9B로 돌려요. 11/18(27B)과 숫자를 직접 비교하지 말고, 예전/새 프롬프트 차이만 보세요.")
print("LLM:", CFG["llm_model"])

def run(cmd, check=True, quiet=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and check:
        print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(f"실패: {cmd}")
    if not quiet and r.stdout.strip():
        print(r.stdout.strip()[-1500:])
    return r

In [ ]:
# vLLM 설치 + 모델 다운로드를 백그라운드로 시작 (11/18 노트북과 같음). 그동안 아래 셀을 계속 실행하세요
VLLM_ENV = Path("/content/vllm-env")
SETUP_LOG = Path("/content/llm_setup.log")
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    pass
setup_script = f'''set -e
echo "[1/2] vLLM 설치 $(date)"
if [ ! -x {VLLM_ENV}/bin/vllm ]; then
  pip install -q uv
  rm -rf {VLLM_ENV}
  uv venv {VLLM_ENV} --python 3.12 -q
  uv pip install --python {VLLM_ENV}/bin/python -q "vllm>=0.17.0" --torch-backend=auto
fi
echo "[2/2] 모델 다운로드 {CFG['llm_model']} $(date)"
{"export HF_TOKEN=" + HF_TOKEN if HF_TOKEN else ""}
{VLLM_ENV}/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('{CFG['llm_model']}')"
echo "SETUP_DONE $(date)"
'''
Path("/content/llm_setup.sh").write_text(setup_script)
if "SETUP_DONE" not in (SETUP_LOG.read_text() if SETUP_LOG.exists() else ""):
    SETUP_PROC = subprocess.Popen(f"bash /content/llm_setup.sh > {SETUP_LOG} 2>&1", shell=True)
    print("백그라운드 시작. 진행 상황:", SETUP_LOG)

In [ ]:
# backend: 새 프롬프트 브랜치(fix/refusal-prompt). main 에 합쳐져 브랜치가 지워졌으면 main
NEW_BRANCH = "fix/refusal-prompt"
has_branch = bool(run(f"git ls-remote --heads https://github.com/Jaeyong-s-HR-Chatbot/backend.git {NEW_BRANCH}", quiet=True).stdout.strip())
for repo, branch in (("backend", NEW_BRANCH if has_branch else "main"), ("DB", "main"), ("RAG-hyerim", "main")):
    d = ROOT / repo
    if d.exists():
        shutil.rmtree(d)
    depth = "" if repo == "backend" else "--depth 1"     # backend 는 예전 프롬프트 커밋을 꺼내야 해서 전체
    run(f"git clone -q {depth} -b {branch} https://github.com/Jaeyong-s-HR-Chatbot/{repo}.git {d}")
    print(f"{repo:11s} {branch:20s}", run(f"git -C {d} log -1 --format='%h %ad %s' --date=short", quiet=True).stdout.strip())
BACKEND, DB, HYERIM = ROOT / "backend", ROOT / "DB", ROOT / "RAG-hyerim"

# 예전 프롬프트 = 고치기 직전 backend 커밋(6d9edba, 11/18 평가 때 프롬프트)의 SYSTEM_PROMPT
OLD_COMMIT = "6d9edba"
old_src = run(f"git -C {BACKEND} show {OLD_COMMIT}:app/graph/nodes.py", quiet=True).stdout
OLD_PROMPT = re.search(r'SYSTEM_PROMPT = "{3}(.*?)"{3}', old_src, re.S).group(1)

os.environ.update({"BACKEND_DIR": str(BACKEND), "HRDB_URL": "", "SEARCH_MODE": "keyword", "LLM_BASE_URL": "",
                   "PSEUDO_KEY": "colab-notebook-only", "CHAT_LOG_DB": "false", "RULES_PATH": str(DB / "rules/rules.json")})
spec = importlib.util.spec_from_file_location("eval_golden", HYERIM / "scripts/eval_golden.py")
eg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eg)
sys.path.insert(0, str(BACKEND))
from app import config as be_config
from app.graph import nodes as be_nodes, retriever as be_retriever
NEW_PROMPT = be_nodes.SYSTEM_PROMPT
PROMPTS = {"예전": OLD_PROMPT, "새": NEW_PROMPT}
assert OLD_PROMPT != NEW_PROMPT, "프롬프트가 같아요 (브랜치 확인)"
for k, p in PROMPTS.items():
    print(f"--- {k} 규칙 2·3")
    print("\n".join(l for l in p.splitlines() if l[:2] in ("2.", "3.")))

ITEMS = eg.load_items()
SPLIT = eg.get_splits(ITEMS)
for it in ITEMS:
    it["split"] = SPLIT[it["id"]]
# 퇴직연금 골든셋 20문항 (eval/golden/pension, 2026-11-19 추가). split 은 파일에 들어 있음
PENSION = json.load(open(HYERIM / "eval/golden/pension/pension_golden_set.json", encoding="utf-8"))["items"]
_main_load = eg.load_items
eg.load_items = lambda: _main_load() + PENSION        # cmd_score 가 퇴직연금 문항도 찾게
Q = [it for it in ITEMS if it["split"] in ("dev", "test") and not it["needs_hr_data"] and not it.get("calculation")]
# 퇴직연금: A 규정 · D 거절/틀린 전제만 (B·C는 본인 퇴직연금 데이터가 필요한데 DB 저장소에 아직 없음, PN-P1-08 타인 조회는 의도 분류가 거름)
QP = [it for it in PENSION if it["split"] in ("dev", "test") and it["kind"] in ("A", "D") and it["id"] != "PN-P1-08"]
Q += QP
print(len(Q), "문항", dict(Counter(it["type"] for it in Q)), f"(퇴직연금 {len(QP)}: {[it['id'] for it in QP]})")

In [ ]:
# ② 하이브리드 검색 + ③ 리랭커 (11/18 노트북 코드·값 그대로, 예전 조문 단위 조각) → oracle_context 의 오답 조문·규정 없음 근거
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

def _cite(r):
    art, t = (r.get("article") or "").strip(), (r.get("article_title") or "").strip()
    return f"{r['doc_name']} {art}" + (f"({t[:20]})" if t and art.startswith("제") else "")

CHUNKS = []
for path in eg.CORPUS_FILES:
    for line in open(path, encoding="utf-8"):
        r = json.loads(line)
        if not r.get("searchable"):
            continue
        CHUNKS.append({"id": r["chunk_id"], "doc_id": r["document_id"], "doc_name": r["doc_name"],
                       "doc_kind": r.get("doc_kind") or ("법령" if r["chunk_id"].startswith("L") else "내부규정"),
                       "article": (r.get("article") or "").strip(), "article_title": (r.get("article_title") or "").strip(),
                       "citation": _cite(r), "revised": r.get("effective_date") or "", "text": r["text"],
                       "embed_text": f"{r['doc_name']} {r.get('article') or ''} {r.get('article_title') or ''}".strip() + "\n" + r["text"]})

def _load(cls, name, **kw):
    try:
        return cls(name, device=DEVICE, model_kwargs={"torch_dtype": torch.float16 if HAS_GPU else torch.float32}, **kw)
    except TypeError:
        m = cls(name, device=DEVICE, **kw)
        if HAS_GPU:
            m.model.half() if hasattr(m, "model") else m.half()
        return m

kiwi = Kiwi()
_KEEP = {"NNG", "NNP", "NR", "SL", "SN", "SH", "XR", "VV", "VA"}
def ko_tokens(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)
    return [t.form + ("다" if t.tag in ("VV", "VA") else "") for t in kiwi.tokenize(text) if t.tag in _KEEP]

BM25 = BM25Okapi([ko_tokens(c["embed_text"]) for c in CHUNKS])
EMBEDDER = _load(SentenceTransformer, CFG["embed_model"])
EMBEDDER.max_seq_length = 1024
DOC_EMB = EMBEDDER.encode([c["embed_text"] for c in CHUNKS], batch_size=32, normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
RERANKER = _load(CrossEncoder, CFG["rerank_model"], max_length=1024)

def bm25_rank(query, n):
    s = BM25.get_scores(ko_tokens(query)); top = np.argsort(-s)[:n]
    return [int(i) for i in top if s[i] > 0]

def dense_rank(query, n):
    v = EMBEDDER.encode([re.sub(r"\[[^\]]*\]", " ", query)], normalize_embeddings=True, convert_to_numpy=True)[0]
    return [int(i) for i in np.argsort(-(DOC_EMB @ v))[:n]]

def fuse(lists, weights, n):
    sc = defaultdict(float)
    for lst, w in zip(lists, weights):
        for r, i in enumerate(lst):
            sc[i] += w / (CFG["rrf_c"] + r + 1)
    for i in sc:
        if CHUNKS[i]["doc_kind"] == "법령":
            sc[i] *= CFG["law_weight"]
    return sorted(sc, key=sc.get, reverse=True)[:n]

def search(question, n=None):
    n = n or CFG["k_candidates"]
    bq = be_retriever.rewrite(question) if CFG["rewrite_bm25"] else question
    cand = fuse([bm25_rank(bq, max(CFG["k_bm25"], n)), dense_rank(question, max(CFG["k_dense"], n))],
                [CFG["bm25_weight"], CFG["dense_weight"]], n)
    s = np.asarray(RERANKER.predict([(question, CHUNKS[i]["embed_text"]) for i in cand], batch_size=16,
                                    convert_to_numpy=True, show_progress_bar=False), dtype=float).reshape(-1)
    if s.min() < 0 or s.max() > 1:
        s = 1 / (1 + np.exp(-s))
    scored = [(i, min(1.0, v + (CFG["internal_bonus"] if CHUNKS[i]["doc_kind"] == "내부규정" else 0.0))) for i, v in zip(cand, s)]
    return sorted(scored, key=lambda x: -x[1])

RETRIEVER = SimpleNamespace(search=lambda q, k=15: [{x: CHUNKS[i][x] for x in ("id", "citation", "revised", "text")}
                                                    for i, _ in search(q)[:k]])
TEXTS = eg.chunk_texts()
t0 = time.time()
CTX = {it["id"]: eg.oracle_context(it, TEXTS, RETRIEVER) for it in Q}
print(f"근거 {len(CTX)}문항 {time.time() - t0:.0f}초 · 예: {Q[0]['id']} → {[c['id'] for c in CTX[Q[0]['id']]]}")
del EMBEDDER, RERANKER
torch.cuda.empty_cache()      # vLLM에 GPU를 넘김

In [ ]:
# vLLM 켜기 (11/18 노트북과 같은 옵션)
LLM_LOG = Path("/content/vllm.log")
LLM_API_KEY = hashlib.sha256(os.urandom(16)).hexdigest()[:32]
LLM_BASE_URL = f"http://127.0.0.1:{CFG['llm_port']}/v1"
while "SETUP_DONE" not in SETUP_LOG.read_text():
    if "SETUP_PROC" in globals() and SETUP_PROC.poll() not in (None, 0):
        print(SETUP_LOG.read_text()[-3000:]); raise RuntimeError("vLLM 설치·다운로드 실패 (로그 위)")
    print("설치·다운로드 기다리는 중…", (SETUP_LOG.read_text().strip().splitlines() or ["시작 전"])[-1][:100]); time.sleep(30)
run("pkill -f 'vllm serve' || true", check=False, quiet=True)
time.sleep(3)
help_text = run(f"{VLLM_ENV}/bin/vllm serve --help=all", check=False, quiet=True).stdout
free, total = torch.cuda.mem_get_info()
small = GPU_GB < 39
util = round(min(0.9, (free - 1 * 1024**3) / total), 3)
max_len = 16384 if GPU_GB >= 75 else 8192
cmd = [str(VLLM_ENV / "bin/vllm"), "serve", CFG["llm_model"], "--served-model-name", CFG["served_name"],
       "--host", "127.0.0.1", "--port", str(CFG["llm_port"]), "--api-key", LLM_API_KEY,
       "--max-model-len", str(max_len), "--gpu-memory-utilization", str(util),
       "--max-num-seqs", str(32 if GPU_GB >= 75 else 8), "--reasoning-parser", "qwen3", "--trust-remote-code",
       "--limit-mm-per-prompt", json.dumps({"image": 0, "video": 0})]
if small:
    cmd += ["--quantization", "fp8", "--enforce-eager"]
if "--default-chat-template-kwargs" in help_text:
    cmd += ["--default-chat-template-kwargs", json.dumps({"enable_thinking": CFG["thinking"]})]
print(" ".join(cmd).replace(LLM_API_KEY, "***"))
LLM_PROC = subprocess.Popen(cmd, stdout=open(LLM_LOG, "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, **({"HF_TOKEN": HF_TOKEN} if HF_TOKEN else {})})
import urllib.request
t0 = time.time()
while True:
    try:
        m = json.loads(urllib.request.urlopen(urllib.request.Request(f"{LLM_BASE_URL}/models", headers={"Authorization": f"Bearer {LLM_API_KEY}"}), timeout=5).read())
        print(f"✅ vLLM 준비 ({time.time() - t0:.0f}초) · {m['data'][0].get('root', '')}"); break
    except Exception:
        if LLM_PROC.poll() is not None:
            print("\n".join([l for l in LLM_LOG.read_text().splitlines() if "Error" in l][-8:])); raise RuntimeError("vLLM이 꺼졌어요")
        time.sleep(15)

from openai import OpenAI
LLM = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY, timeout=300, max_retries=2)
be_config.LLM_BASE_URL, be_config.LLM_MODEL, be_config.LLM_API_KEY = LLM_BASE_URL, CFG["served_name"], LLM_API_KEY
be_config.LLM_MAX_TOKENS, be_config.LLM_TIMEOUT, be_config.USE_FAKE_LLM = CFG["llm_max_tokens"], 300.0, False

def llm_chat(messages, max_tokens=None, json_schema=None):
    kw = {"response_format": {"type": "json_schema", "json_schema": {"name": "out", "schema": json_schema}}} if json_schema else {}
    r = LLM.chat.completions.create(model=CFG["served_name"], messages=messages, temperature=0.0,
                                    max_tokens=max_tokens or CFG["llm_max_tokens"],
                                    extra_body={"chat_template_kwargs": {"enable_thinking": CFG["thinking"]}}, **kw)
    return be_nodes._clean(r.choices[0].message.content or "")
print(llm_chat([{"role": "user", "content": "한 문장으로 자기소개 해 줘."}], max_tokens=60))

In [ ]:
# 답변 생성 (backend nodes._llm_answer 그대로, SYSTEM_PROMPT만 바꿈) → 판정 → eval_golden.cmd_score
JUDGE_SCHEMA = json.loads(json.dumps(eg.JUDGE_SCHEMA))
JUDGE_SCHEMA["properties"]["declined_properly"] = {"anyOf": [{"type": "boolean"}, {"type": "null"}]}

def qwen_judge(it, answer):
    user = json.dumps({"문항유형": it["type"], "질문": it["question"], "정답": it["reference_answer"], "근거규정": it["basis"],
                       "key_facts": it["key_facts"], "must_not_include": it["must_not_include"], "챗봇답변": answer},
                      ensure_ascii=False, indent=1)
    for _ in range(2):
        try:
            j = json.loads(llm_chat([{"role": "system", "content": eg.JUDGE_SYSTEM}, {"role": "user", "content": user}],
                                    max_tokens=1500, json_schema=JUDGE_SCHEMA))
            if len(j["key_facts"]) == len(it["key_facts"]) and len(j["must_not_include"]) == len(it["must_not_include"]):
                return j
        except Exception:
            pass
    return None

def answer_one(it):
    s = {"intent": "rule", "role": "employee", "masked_question": it["question"], "chunks": CTX[it["id"]], "lookup": {}, "calc": {}}
    t0 = time.time()
    try:
        ans, err = be_nodes._llm_answer(s), None
    except Exception as e:
        ans, err = "", f"{type(e).__name__}: {str(e)[:200]}"
    out = {"answer": ans, "kind": "answer", "chunks": [c["id"] for c in s["chunks"]]}
    out |= be_nodes.normalize_no_evidence({**s, "answer": ans})
    out["chunks"] = [c if isinstance(c, str) else c["id"] for c in out["chunks"]]
    return out, err, int((time.time() - t0) * 1000)

RUNS = {}
for name, prompt in PROMPTS.items():
    be_nodes.SYSTEM_PROMPT = prompt
    t0 = time.time()
    with ThreadPoolExecutor(CFG["llm_concurrency"]) as ex:
        res = list(ex.map(answer_one, Q))
    rows = [{"id": it["id"], "split": it["split"], "mode": "oracle", "model": CFG["served_name"], "ms": ms, "error": err,
             "answer": r["answer"], "kind": r["kind"], "intent": "rule", "tools_used": None, "chunks": r["chunks"],
             "context_text": "\n".join(c["text"] for c in CTX[it["id"]])}
            for it, (r, err, ms) in zip(Q, res)]
    run_dir = eg.new_run(f"prompt_{'old' if name == '예전' else 'new'}_{CFG['served_name']}")
    eg.write_jsonl(run_dir / "answers.jsonl", rows)
    t1 = time.time()
    with ThreadPoolExecutor(CFG["llm_concurrency"]) as ex:
        js = list(ex.map(lambda a: qwen_judge(*a) if a[1] else None, [(it, r["answer"]) for it, r in zip(Q, rows)]))
    eg.write_jsonl(run_dir / "judged.jsonl", [{"id": r["id"], "judge": j} for r, j in zip(rows, js)])
    eg.cmd_score(SimpleNamespace(run=str(run_dir), judge=False))
    RUNS[name] = {"dir": run_dir, "rows": rows, "summary": json.loads((run_dir / "summary.json").read_text(encoding="utf-8")),
                  "scored": pd.DataFrame(eg.read_jsonl(run_dir / "judged.jsonl"))}   # cmd_score 가 채점 결과를 judged.jsonl 에 다시 씀
    print(f"{name}: 생성 {t1 - t0:.0f}초 · 판정 {time.time() - t1:.0f}초 → {run_dir.name}")
be_nodes.SYSTEM_PROMPT = NEW_PROMPT

In [ ]:
# 비교
QT = {it["id"]: it for it in Q}
def strict_refused(rows):    # 좁은 기준: 거절로 처리된 답 (규정에서 확인되지 않아요 로 시작)
    return {r["id"]: r["kind"] == "refused" for r in rows}

SHOW = ["accuracy", "key_fact_ratio", "규정없음_거절률", "잘못된_거절률", "틀린전제_교정률", "포괄질문_accuracy",
        "금지주장_위반률", "지어낸숫자_의심률", "근거조문_인용률", "n_scored", "judged"]
T = pd.DataFrame({k: {m: v["summary"].get(m) for m in SHOW} for k, v in RUNS.items()})
for k, v in RUNS.items():
    sr = strict_refused(v["rows"])
    T.loc["잘못된_거절률(좁은 기준)", k] = round(np.mean([sr[i] for i in sr if QT[i]["type"] == "일반"]), 3)
    T.loc["규정없음_거절률(좁은 기준)", k] = round(np.mean([sr[i] for i in sr if QT[i]["type"] == "규정 없음"]), 3)
    T.loc["accuracy dev", k] = v["summary"]["accuracy_by_split"]["dev"]
    T.loc["accuracy test", k] = v["summary"]["accuracy_by_split"]["test"]
print(f"모델: {CFG['llm_model']} · 문항 {len(Q)} (11/18 노션 27B Oracle: 규정없음 거절 0.333 · 잘못된 거절 0.194 · dev accuracy 0.436, 문항 구성이 달라 직접 비교는 참고만)")
display(T)

In [ ]:
# 퇴직연금 문항: 예전 / 새 정답 여부
SC = {k: {r["id"]: r for r in v["scored"].to_dict(orient="records")} for k, v in RUNS.items()}
print("■ 퇴직연금 9문항")
display(pd.DataFrame([{"id": it["id"], "kind": it["kind"], "유형": it["type"], "질문": it["question"][:40],
                       "예전 정답": SC["예전"][it["id"]]["correct"], "새 정답": SC["새"][it["id"]]["correct"]} for it in QP]))

# 규정 없음 10문항: 예전 / 새 답변 나란히
pd.set_option("display.max_colwidth", 200)
A = {k: {r["id"]: r for r in v["rows"]} for k, v in RUNS.items()}
NO = [it for it in Q if it["type"] == "규정 없음"]     # 팀 골든셋 10 + 퇴직연금 2
display(pd.DataFrame([{"id": it["id"], "질문": it["question"],
                       "예전": A["예전"][it["id"]]["answer"][:200], "새": A["새"][it["id"]]["answer"][:200]} for it in NO]))

# 일반 문항 중 새 프롬프트에서 새로 거절된 것 (잘못된 거절이 늘었나)
flip = [it for it in Q if it["type"] == "일반" and A["새"][it["id"]]["kind"] == "refused" and A["예전"][it["id"]]["kind"] != "refused"]
back = [it for it in Q if it["type"] == "일반" and A["예전"][it["id"]]["kind"] == "refused" and A["새"][it["id"]]["kind"] != "refused"]
print(f"일반 문항: 새로 거절됨 {len(flip)} · 거절이 풀림 {len(back)}")
display(pd.DataFrame([{"id": it["id"], "질문": it["question"], "정답": it["reference_answer"][:120],
                       "예전 답": A["예전"][it["id"]]["answer"][:200]} for it in flip]))

In [ ]:
from google.colab import files
out = Path(f"/content/prompt_refusal_{CFG['served_name']}.zip")
with __import__("zipfile").ZipFile(out, "w") as z:
    for k, v in RUNS.items():
        for f in v["dir"].iterdir():
            z.write(f, f"{v['dir'].name}/{f.name}")
    z.writestr("compare.json", json.dumps({"model": CFG["llm_model"], "table": T.to_dict(), "prompts": PROMPTS},
                                          ensure_ascii=False, indent=1, default=str))
files.download(str(out))